# Phase 2: Data Preprocessing

**Project:** AI Mental Health Assessment Platform  
**Goal:** Prepare the dataset for 3-in-1 prediction:

1. Mental Health Condition Prediction
2. Severity Level Prediction
3. Treatment Recommendation

This notebook creates clean, encoded, scaled, train/test-ready data for Phase 3 model training.

## Step 1: Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import joblib

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)

## Step 2: Define Project Paths

The notebook supports local repo structure and Kaggle input structure.

In [ ]:
ROOT_DIR = Path("..").resolve()
DATA_DIR = ROOT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = ROOT_DIR / "models"
REPORTS_DIR = ROOT_DIR / "reports"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

possible_paths = [
    DATA_DIR / "mental_health_prediction.csv",
    Path("/kaggle/input/mental-health-prediction/mental_health_prediction.csv"),
    Path("/kaggle/input/mental-health-prediction-dataset/mental_health_prediction.csv"),
    Path("/kaggle/input") / "mental_health_prediction.csv",
]

DATA_PATH = None
for path in possible_paths:
    if path.exists():
        DATA_PATH = path
        break

if DATA_PATH is None:
    raise FileNotFoundError("Dataset not found. Please place mental_health_prediction.csv inside data/ folder.")

DATA_PATH

## Step 3: Load Dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()

## Step 4: Dataset Shape

In [ ]:
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

## Step 5: Identify Target Columns

In [ ]:
TARGET_COLUMNS = ["mental_health_condition", "severity", "treatment"]

for target in TARGET_COLUMNS:
    print(target, "->", df[target].nunique(), "classes")
    print(df[target].value_counts())
    print("-" * 80)

## Step 6: Define Feature Columns

We remove all 3 target columns from the input features to avoid data leakage.

In [ ]:
feature_cols = [col for col in df.columns if col not in TARGET_COLUMNS]
feature_cols

## Step 7: Separate Numerical and Categorical Features

In [ ]:
numerical_cols = df[feature_cols].select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = df[feature_cols].select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numerical Features:", numerical_cols)
print("Categorical Features:", categorical_cols)

## Step 8: Missing Value Summary

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values("missing_count", ascending=False)

missing_summary

## Step 9: Duplicate Record Check

In [ ]:
duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

## Step 10: Remove Duplicate Records

In [ ]:
df_clean = df.drop_duplicates().reset_index(drop=True)
print("Before:", df.shape)
print("After :", df_clean.shape)

## Step 11: Outlier Detection Using IQR

In [ ]:
def iqr_outlier_summary(dataframe, cols):
    rows = []
    for col in cols:
        q1 = dataframe[col].quantile(0.25)
        q3 = dataframe[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outlier_count = ((dataframe[col] < lower) | (dataframe[col] > upper)).sum()
        rows.append({
            "feature": col,
            "lower_bound": lower,
            "upper_bound": upper,
            "outlier_count": int(outlier_count),
            "outlier_percentage": round(outlier_count / len(dataframe) * 100, 2)
        })
    return pd.DataFrame(rows).sort_values("outlier_count", ascending=False)

outlier_summary = iqr_outlier_summary(df_clean, numerical_cols)
outlier_summary

## Step 12: Visualize Outliers Before Clipping

In [ ]:
plt.figure(figsize=(14, 6))
sns.boxplot(data=df_clean[numerical_cols])
plt.xticks(rotation=75)
plt.title("Numerical Feature Outliers Before Clipping")
plt.show()

## Step 13: Clip Outliers Using IQR

For this dataset, clipping is safer than removing rows because the dataset has only 500 records.

In [ ]:
def clip_outliers_iqr(dataframe, cols, factor=1.5):
    dataframe = dataframe.copy()
    for col in cols:
        q1 = dataframe[col].quantile(0.25)
        q3 = dataframe[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - factor * iqr
        upper = q3 + factor * iqr
        dataframe[col] = dataframe[col].clip(lower=lower, upper=upper)
    return dataframe

df_clean = clip_outliers_iqr(df_clean, numerical_cols)
df_clean.head()

## Step 14: Visualize Outliers After Clipping

In [ ]:
plt.figure(figsize=(14, 6))
sns.boxplot(data=df_clean[numerical_cols])
plt.xticks(rotation=75)
plt.title("Numerical Feature Outliers After Clipping")
plt.show()

## Step 15: Build Preprocessing Pipeline

Numerical columns:
- Missing values: Median
- Scaling: StandardScaler

Categorical columns:
- Missing values: Most frequent
- Encoding: OneHotEncoder

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numerical_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

preprocessor

## Step 16: Helper Function for Feature Names

In [ ]:
def get_processed_feature_names(fitted_preprocessor, numerical_cols, categorical_cols):
    feature_names = list(numerical_cols)
    if categorical_cols:
        encoder = fitted_preprocessor.named_transformers_["cat"].named_steps["encoder"]
        encoded_names = encoder.get_feature_names_out(categorical_cols).tolist()
        feature_names.extend(encoded_names)
    return feature_names

## Step 17: Create Preprocessed Data for One Target

In [ ]:
def create_preprocessed_dataset(dataframe, target_col, test_size=0.2):
    X = dataframe[feature_cols]
    y = dataframe[target_col]

    stratify = y if y.nunique() > 1 and y.value_counts().min() >= 2 else None

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=RANDOM_STATE,
        stratify=stratify
    )

    current_preprocessor = ColumnTransformer(transformers=[
        ("num", numeric_pipeline, numerical_cols),
        ("cat", categorical_pipeline, categorical_cols)
    ])

    X_train_processed = current_preprocessor.fit_transform(X_train)
    X_test_processed = current_preprocessor.transform(X_test)

    processed_feature_names = get_processed_feature_names(
        current_preprocessor,
        numerical_cols,
        categorical_cols
    )

    X_train_processed = pd.DataFrame(X_train_processed, columns=processed_feature_names, index=X_train.index)
    X_test_processed = pd.DataFrame(X_test_processed, columns=processed_feature_names, index=X_test.index)

    return {
        "target": target_col,
        "X_train_raw": X_train,
        "X_test_raw": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "X_train_processed": X_train_processed,
        "X_test_processed": X_test_processed,
        "preprocessor": current_preprocessor,
        "feature_names": processed_feature_names
    }

## Step 18: Preprocess Data for Target 1 — Mental Health Condition

In [ ]:
condition_data = create_preprocessed_dataset(df_clean, "mental_health_condition")

print("X_train:", condition_data["X_train_processed"].shape)
print("X_test :", condition_data["X_test_processed"].shape)
print("y_train distribution:")
print(condition_data["y_train"].value_counts())

## Step 19: Preprocess Data for Target 2 — Severity

In [ ]:
severity_data = create_preprocessed_dataset(df_clean, "severity")

print("X_train:", severity_data["X_train_processed"].shape)
print("X_test :", severity_data["X_test_processed"].shape)
print("y_train distribution:")
print(severity_data["y_train"].value_counts())

## Step 20: Preprocess Data for Target 3 — Treatment

In [ ]:
treatment_data = create_preprocessed_dataset(df_clean, "treatment")

print("X_train:", treatment_data["X_train_processed"].shape)
print("X_test :", treatment_data["X_test_processed"].shape)
print("y_train distribution:")
print(treatment_data["y_train"].value_counts())

## Step 21: Check Processed Data Quality

In [ ]:
for name, data in {
    "condition": condition_data,
    "severity": severity_data,
    "treatment": treatment_data
}.items():
    print(f"\n{name.upper()}")
    print("Train missing:", data["X_train_processed"].isnull().sum().sum())
    print("Test missing :", data["X_test_processed"].isnull().sum().sum())
    print("Train shape  :", data["X_train_processed"].shape)
    print("Test shape   :", data["X_test_processed"].shape)

## Step 22: Save Clean Dataset

In [ ]:
clean_data_path = PROCESSED_DIR / "mental_health_clean.csv"
df_clean.to_csv(clean_data_path, index=False)
clean_data_path

## Step 23: Save Train/Test Data for All 3 Targets

In [ ]:
all_target_data = {
    "condition": condition_data,
    "severity": severity_data,
    "treatment": treatment_data
}

for short_name, data in all_target_data.items():
    target_dir = PROCESSED_DIR / short_name
    target_dir.mkdir(parents=True, exist_ok=True)

    data["X_train_processed"].to_csv(target_dir / "X_train.csv", index=False)
    data["X_test_processed"].to_csv(target_dir / "X_test.csv", index=False)
    data["y_train"].to_csv(target_dir / "y_train.csv", index=False)
    data["y_test"].to_csv(target_dir / "y_test.csv", index=False)

    data["X_train_raw"].to_csv(target_dir / "X_train_raw.csv", index=False)
    data["X_test_raw"].to_csv(target_dir / "X_test_raw.csv", index=False)

    print(f"Saved: {target_dir}")

## Step 24: Save Fitted Preprocessors

In [ ]:
for short_name, data in all_target_data.items():
    joblib.dump(data["preprocessor"], MODELS_DIR / f"{short_name}_preprocessor.pkl")
    print(f"Saved: {MODELS_DIR / f'{short_name}_preprocessor.pkl'}")

## Step 25: Save Metadata

In [ ]:
metadata = {
    "random_state": RANDOM_STATE,
    "target_columns": TARGET_COLUMNS,
    "feature_columns": feature_cols,
    "numerical_columns": numerical_cols,
    "categorical_columns": categorical_cols,
    "processed_feature_count": len(condition_data["feature_names"]),
    "processed_feature_names": condition_data["feature_names"],
    "clean_dataset_shape": list(df_clean.shape),
    "train_test_split": "80/20 stratified split"
}

metadata_path = PROCESSED_DIR / "preprocessing_metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)

metadata_path

## Step 26: Final Preprocessing Summary

In [ ]:
summary_rows = []
for short_name, data in all_target_data.items():
    summary_rows.append({
        "target_project": short_name,
        "target_column": data["target"],
        "train_rows": data["X_train_processed"].shape[0],
        "test_rows": data["X_test_processed"].shape[0],
        "processed_features": data["X_train_processed"].shape[1],
        "target_classes": data["y_train"].nunique()
    })

preprocessing_summary = pd.DataFrame(summary_rows)
preprocessing_summary

## Step 27: Save Preprocessing Summary Report

In [ ]:
summary_path = REPORTS_DIR / "preprocessing_summary.csv"
preprocessing_summary.to_csv(summary_path, index=False)
summary_path

## Step 28: Conclusion

Phase 2 is completed.

The dataset is now ready for Phase 3 model training. We prepared three ML-ready datasets for:

1. Mental health condition prediction
2. Severity prediction
3. Treatment recommendation

Next step: **Phase 3 — Train 3 Machine Learning Models**.